# Agentic human activity recognition
This walkthrough classifies the **current** MuRAL sensor event using exactly ten preceding events. The BiLSTM with self-attention summarizes the previous window. The LangGraph agent retrieves household context and uses it alongside sensor evidence and model scores to select one of 22 activity IDs.

The example runs locally with synthetic sensor data, a small in-memory knowledge store, and a deterministic stand-in for Claude. It uses the original trained model checkpoint. See the last section for live services and data replay.

## 1. Prepare one recognition request
History runs from oldest to newest. Each row contains a time, an activated sensor, and its status. In live use, the optional history labels can come from earlier agent decisions. Keep the current event's true activity separate for scoring.

In [ ]:
from agentic_har.demo import EXAMPLE, build_demo_pipeline
from agentic_har.pipeline import encode_history

print(f"Household: {EXAMPLE.household_id}")
print(f"History length: {len(EXAMPLE.history)}")
EXAMPLE.event.model_dump()

## 2. Encode history and run the BiLSTM
The original encoder represents time of day with sine and cosine, applies a scaled log transform to elapsed seconds, and one-hot encodes the sensor categories and status. The checkpoint predicts the activity at the last event **inside** this window. The agent uses those scores as context for the current event.

In [ ]:
pipeline = build_demo_pipeline()
feature_window = encode_history(EXAMPLE.history)
print(f"Feature matrix: {feature_window.shape}")
pipeline.predictor.predict_activity(feature_window)

## 3. Run retrieval and the recognition graph
The graph retrieves layout and sensor information plus historical patterns filtered to the current household. It sends that context, the sensor evidence, and the three model scores to a reasoner with a structured output schema. The local stand-in demonstrates the data flow. Its output is a scripted example and cannot measure recognition quality.

In [ ]:
response = pipeline.recognize(EXAMPLE)
response.model_dump()

## 4. Replay MuRAL and evaluate
Point `MURAL_CSV` to the original `MuRAL_final_preprocessed.csv`. Replay sorts one house by event index and builds overlapping windows within that house. The historical labels are omitted by default to approximate deployment. Set `labeled_history=True` only when explicitly reproducing the original experiment's use of annotated history.

The offline reasoner is suitable for checking that replay runs. For real evaluation, seed Pinecone and configure the live services as described in the README, then enable `USE_LIVE`. Match the retrieval household to the replay data. The original experiment replayed house 01 with H002 context. House 01 is in the bundled checkpoint’s training split, so this replay is illustrative. For a generalization study, use evaluation records independent of both training and indexed histories.

In [ ]:
from itertools import islice
from pathlib import Path

import pandas as pd

from agentic_har.dataset import replay_examples
from agentic_har.metrics import compute_metrics
from agentic_har.runtime import build_live_pipeline

MURAL_CSV = None  # Set to Path("/path/to/MuRAL_final_preprocessed.csv")
USE_LIVE = False
HOUSE, HOUSEHOLD_ID = "01", "H002"

if MURAL_CSV is not None:
    runner = build_live_pipeline() if USE_LIVE else pipeline
    examples = replay_examples(Path(MURAL_CSV), HOUSE, HOUSEHOLD_ID)
    results = pd.DataFrame(
        [(truth, runner.recognize(request).label) for request, truth in islice(examples, 5)],
        columns=["actual", "predicted"],
    )
    summary = compute_metrics(results.actual, results.predicted, len(results), 0.0)
    display(results, pd.Series({key: value for key, value in summary.items() if key != "report"}))
else:
    print("Set MURAL_CSV to run the optional replay.")

## 5. Serve the same pipeline
The API exposes `POST /v1/recognize`. The request body matches `EXAMPLE.model_dump(mode="json")` and the response matches the schema used above. Follow the README to start the service and send `example_request.json` or try the interactive `/docs` page.